# K-Means Clustering

## Unsupervised learning refresher

As we saw in previous lectures, [Unsupervised learning](https://en.wikipedia.org/wiki/Unsupervised_learning) is a type of machine learning where the model is trained on data that has not been labeled, classified, or categorized. Unlike supervised learning, where the model is trained on a dataset with known outputs, unsupervised learning algorithms must find patterns and relationships within the data on their own. This makes unsupervised learning particularly useful for exploratory data analysis, where the goal is to discover hidden structures or groupings within the data.

Unsupervised learning problems can be categorized into several types, each addressing different aspects of data analysis and pattern discovery. Here are the main categories of unsupervised learning problems:
- **Clustering**: a problem where entry data has to be divided into categories/groups based on common features
- **Dimensionality reduction**: this problem involves transforming an original dataset into one with fewer features (dimensions). One well-known method is [Principal Component Analysis](https://en.wikipedia.org/wiki/Principal_component_analysis) (PCA). The main purpose of PCA is to merge existing features into new ones (known as Principal Components in PCA) with the aim to create a more manageable dataset while retaining as much of the original information as possible. Principal components are inherently independent (uncorrelated). A dataset with uncorrelated features is often a requirement for certain machine learning algorithms. Therefore, dimensionality reduction techniques can be used to preprocess data for machine learning applications.
- **Anomaly detection**: a problem where outliers are found in a sample. It is used mostly to detect bank frauds
- **Association**: a problem where data is used to identify rules that describe large portions of the data. For example, [Market Basket Analysis](https://www.geeksforgeeks.org/data-science/market-basket-analysis-in-data-mining/) uses association rules to find products that are frequently bought together.

## Definition

*K-Means Clustering* is an unsupervised learning algorithm that partitions data points into distinct groups (also known
as **clusters**) based on their similarity with each other. The number of distinct groups is determined by the value **k**.

## Step-wise approach

The algorithm is composed of the following steps. Let us consider an example where a given dataset contains eight points.

![step_0.png](./images/step_0.png)

### Step 1

Given a value of **k**, the code selects **k** randomly placed points (called **centroids**). 

![step_1.png](./images/step_1.png)

### Step 2

Group together all points which are nearest to a centroid. As you can see from the image below, this may result in unsatisfying grouping.

![step_2.png](./images/step_2.png)

### Step 3

Compute the mean position of all points in a cluster. The image below shows this procedure for the orange points.

![step_3.png](./images/step_3.png)

### Step 4

Reposition the centroids to these new mean-positions.

![step_4.png](./images/step_4.png)

### Step 5

Re-assign each data point to the nearest centroid. 

![step_5.png](./images/step_5.png)

At this point, repeat **Step 3**, **Step 4** and **Step 5** until no data point changes cluster.

## How to choose the number of clusters


The short answer is that no choice is a-priori right or wrong. This is because we are dealing with unsupervised learning. There are just patterns/trends we are trying to extract from the data and interpret.

It depends on the available data and the use case. For example if you are asked to segment the data of customers for a marketing
survey, having too many clusters may not be the best. In fact, the business department may find this analysis too complex to understand or use. There are however some numerical techniques that can support us in this choice. One of them is the **Within Cluster Sum of Squares (WCSS)**.

### Within Cluster Sum of Squares

The algorithm minimizes the sum of the squared Euclidean distances among the data points of the cluster centroids.  The value of the sum will decrease as the number of centroids **k** increases and reaches zero for **k** equals the number of data points. The WCSS for the eight points examples considered before looks something like

![elbow_plot.png](./images/elbow_plot.png)


The rate at which the WCSS decreases, diminishes as k increases. This is why such a plot is usually referred to as **Elbow plot**. The value of k which is usually taken is the one where the curve makes the turn (the elbow), as before it we see a sharp decline and after a small one. In other words, adding more centroids does not produce a significant improvement in the clustering. In our example, the elbow is at `k=2`.

Other scores you can explore and use for your analyses include the [Calinsky-Harabasz score](https://en.wikipedia.org/wiki/Calinski%E2%80%93Harabasz_index) and the [Silhouette score](https://en.wikipedia.org/wiki/Silhouette_(clustering).). Analysts often calculate multiple scores to better guide their decision. All of the above can be performed via scikit-learn, you can find more details in its detailed documentation ([link](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html) and [link](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.calinski_harabasz_score.html))

## The importance of feature scaling

K-Means clustering is a *distance-based* algorithm. Such algorithms benefit greatly from scaling all variables to the same scale.The best approach is to **normalize** the values to the interval $[0,1]$. In this way, distances are comparable. This can be done with the fomula

$ x_{normalized} = \frac{x-minimum}{maximum-minimum}$.

## How to implement k-means cluster in Python

The sub-directory `data/` contains a csv file (`input_data.csv`) with 200 data points. Let us load them into a pandas dataframe and plot them.

In [ ]:
import pandas
import matplotlib.pyplot

# Load the input data set
input_dataframe = pandas.read_csv('data/input_data.csv')

# Plot the data set, adding labels and a title
matplotlib.pyplot.figure(figsize=(10, 6))
matplotlib.pyplot.scatter(input_dataframe["x"], input_dataframe["y"], marker="o")
matplotlib.pyplot.title('Input data set')
matplotlib.pyplot.xlabel('x')
matplotlib.pyplot.ylabel('y')

# Show the plot
matplotlib.pyplot.show()

The data set contains three distinct clusters. Let us anyway implement WCSS and check whether the `elbow` sits indeed at `k=3`. First of all, we need to normalize the value to the interval $[0,1]$. For this, we can use the [preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html) package from [scikit-learn](https://scikit-learn.org/stable/index.html).

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# normalize data
scale_norm = MinMaxScaler()
scaled_dataframe = pandas.DataFrame(scale_norm.fit_transform(input_dataframe), columns = input_dataframe.columns)

# Add titles and labels
matplotlib.pyplot.figure(figsize=(10, 6))
matplotlib.pyplot.scatter(scaled_dataframe["x"], scaled_dataframe["y"], marker="o")
matplotlib.pyplot.title('Normalized data set')
matplotlib.pyplot.xlabel('x')
matplotlib.pyplot.ylabel('y')

# Show the plot
matplotlib.pyplot.show()

Let us now implement WCSS.

In [ ]:
from sklearn.cluster import KMeans

### Use WCSS to find a good value k
k_values = list(range(1,20))
wcss_list = []

for k in k_values:
	kmeans = KMeans(n_clusters = k, random_state=42)
	kmeans.fit(scaled_dataframe)
	wcss_list.append(kmeans.inertia_)

matplotlib.pyplot.figure(figsize=(10, 6))
matplotlib.pyplot.plot(k_values, wcss_list)
matplotlib.pyplot.title("Within Cluster Sum of Squares - by k")
matplotlib.pyplot.xlabel("k")
matplotlib.pyplot.ylabel("WCSS Score")
matplotlib.pyplot.tight_layout()
matplotlib.pyplot.show()

We can clearly see that the `elbow`, the value of `k` preceeded by a sharp decline and followed by a small decline, is `k=3`. Let us then apply K-Means clustering to the normalized data set.

In [ ]:
# Instantiate and fit the KMeans model
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(scaled_dataframe[["x", "y"]])

# Add the cluster labels to the DataFrame
scaled_dataframe["cluster"] = kmeans.labels_

# Plot the clusters and centroids
centroids = kmeans.cluster_centers_
clusters = scaled_dataframe.groupby("cluster")

matplotlib.pyplot.figure(figsize=(10, 6))
for cluster, data in clusters:
    matplotlib.pyplot.scatter(data["x"], data["y"], marker="o", label=f'Cluster {cluster}')
    matplotlib.pyplot.scatter(centroids[cluster, 0], centroids[cluster, 1], marker="X", color="black", s=300)

matplotlib.pyplot.xlabel("x")
matplotlib.pyplot.ylabel("y")
matplotlib.pyplot.title("K-Means Clustering")
matplotlib.pyplot.legend()
matplotlib.pyplot.tight_layout()
matplotlib.pyplot.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# Gruppo 1: Età media 20 anni, Reddito medio 20.000€
eta_g1 = np.random.normal(25, 3, 50)
reddito_g1 = np.random.normal(20000, 800, 50)

# Gruppo 2: Età media 60 anni, Reddito medio 22.000€
eta_g2 = np.random.normal(45, 3, 50)
reddito_g2 = np.random.normal(21500, 800, 50)

X = np.vstack((np.column_stack((eta_g1, reddito_g1)), 
               np.column_stack((eta_g2, reddito_g2))))

# 2. K-means SENZA Normalizzazione
kmeans_raw = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_raw = kmeans_raw.fit_predict(X)

# 3. K-means CON Normalizzazione
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans_scaled = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_scaled = kmeans_scaled.fit_predict(X_scaled)

# 4. Plot dei Risultati per il confronto
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: Senza normalizzazione
ax1.scatter(X[:, 0], X[:, 1], c=labels_raw, cmap='viridis', edgecolor='k', s=50)
ax1.set_title("K-means SENZA Normalizzazione\n(Sbagliato: divide in base al Reddito)", fontsize=12)
ax1.set_xlabel("Età (Variazione piccola: 20-60)")
ax1.set_ylabel("Reddito Annuo (Variazione grande: 19k-23k)")

# Grafico 2: Con normalizzazione
ax2.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_scaled, cmap='viridis', edgecolor='k', s=50)
ax2.set_title("K-means CON Normalizzazione\n(Corretto: riconosce i due gruppi Età/Reddito)", fontsize=12)
ax2.set_xlabel("Età (Variazione piccola: 20-60)")
ax2.set_ylabel("Reddito Annuo (Variazione grande: 19k-23k)")

plt.tight_layout()
plt.show()

# When to use clustering

Clustering is a very commonly used technique in EDA, as dividing data in groups can be useful for both visualization/results as well as further analysis. For example, grouping a company's clients by spending to support the creation of customer personas.

Generally speaking, clustering is a relatively easy tool you can often employ in the first few steps of your analysis; however, k-means is a very barebone algorithm that can only sometimes work.

You can find more about performing clustering with SciKit-Learn from their [webpage](https://scikit-learn.org/stable/modules/clustering.html); in particular, let's have a quick look at how different clustering algorithms perform against different scatter distributions:

![comparison](./images/comparison.png)

As you can see, certain algorithms perform better than others, but might be slower (or not!). A good -- slightly more advanced -- clustering algorithm we invite you to explore is [DBSCAN](https://en.wikipedia.org/wiki/DBSCAN)
